## Expérimentation de models

In [10]:
import joblib
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso, Ridge, LinearRegression
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import mean_squared_error, median_absolute_error, r2_score, get_scorer_names

In [18]:
train_data = pd.read_csv("../data/processed_dataset.csv")
test_data = pd.read_parquet("../data/test_data.parquet")

X_train = train_data.drop(
    ["LoyerMensuel_Log1", "IdentifiantMaison"],
    axis=1).select_dtypes(include=["number"])
y_train = train_data["LoyerMensuel_Log1"]

print(X_train)

     Chambres  Superficie_m2  DistanceRoute_m  AgeMaison  LoyerMensuel_BIF  \
0         1.0           46.0            154.0       20.0            199931   
1         4.0          217.0            211.0       23.0            773800   
2         4.0          181.0             64.0       24.0           1182846   
3         6.0          253.0            255.0        2.0           1519435   
4         3.0          162.0             29.0       34.0           1136929   
..        ...            ...              ...        ...               ...   
377       4.0          183.0            213.0       14.0           1524598   
378       6.0          171.0            250.0       34.0           1320592   
379       5.0          223.0            293.0       16.0           2600000   
380       6.0          287.0             24.0       30.0           2600000   
381       6.0          272.0            235.0       31.0           1464786   

     Salon_Bin  SalleDeBainInterieure_Bin  Parking_Bin  Meuble_

### Transformation de données de test

In [21]:
# Les variables numériques
feature_cols = ["AgeMaison", "Quartier_Target", "Indicateur_Confort", "Chambres_par_Superficie", "LoyerMensuel_Log1"]

for col in ['Salon', 'SalleDeBainInterieure', 'Parking', 'Meuble', 'Jardin']:
    test_data[col + "_Bin"] = test_data[col].map({"Oui": 1, "Non": 0}).fillna(0).astype(int)

neighbourhood_encoder = joblib.load("neighbourhood_encoder.joblib") 
test_data["Quartier_Target"] = neighbourhood_encoder.transform(test_data[["Quartier"]])[:, 0]

test_data["LoyerMensuel_Log1"] = np.log1p(test_data["LoyerMensuel_BIF"])

# Création de variables utiles pour le test
# cols_confort = ['Salon_Bin', 'SalleDeBainInterieure_Bin', 'Parking_Bin', 'Meuble_Bin', 'Jardin_Bin']
# test_data['Indicateur_Confort'] = test_data[cols_confort].sum(axis=1)

# test_data['Chambres_par_Superficie'] = test_data['Chambres'] / (test_data['Superficie_m2'] + 0.1)

# Séparation de X_test et y_test
X_test = test_data.drop(
    ["LoyerMensuel_Log1", "IdentifiantMaison"],
    axis=1).select_dtypes(include=["number"])
y_test = test_data["LoyerMensuel_Log1"]

X_test

,Chambres,Superficie_m2,DistanceRoute_m,AgeMaison,LoyerMensuel_BIF,Salon_Bin,SalleDeBainInterieure_Bin,Parking_Bin,Meuble_Bin,Jardin_Bin,Quartier_Target
0,1.0,46.0,154.0,20.0,199931,1,1,0,0,0,0.010428
1,4.0,217.0,211.0,23.0,773800,1,1,0,0,0,0.002128
2,4.0,181.0,64.0,24.0,1182846,1,1,1,0,0,0.002182
3,6.0,253.0,255.0,2.0,1519435,1,1,0,0,1,0.001636
4,3.0,162.0,29.0,34.0,1136929,1,1,1,1,0,0.001636
...,...,...,...,...,...,...,...,...,...,...,...
377,4.0,183.0,213.0,14.0,1524598,1,1,0,0,1,0.002163
378,6.0,171.0,250.0,34.0,1320592,1,1,0,0,0,0.001636
379,5.0,223.0,293.0,16.0,2600000,1,1,0,0,1,0.002061
380,6.0,287.0,24.0,30.0,2600000,1,1,1,0,1,0.001636


In [22]:
def formated_time(second):
    m, s = divmod(second, 60)

    if m > 0:
         f"{int(m)} min {s:.4f} secondes"
    else:
        return f"{s:.4f} secondes"

scoring = ["neg_mean_squared_error", "neg_median_absolute_error", "neg_root_mean_squared_error", "r2"]

### Modèle de Régression Linéaire

In [23]:
linear_model = LinearRegression()

linear_scores = cross_validate(linear_model, X_train, y_train, scoring=scoring)

print(f"Fit Time : {linear_scores["fit_time"]}")
print()
print(f"Score Time : {linear_scores["score_time"]}")
print()
print(f"Test Neg Median Abosulte Error : {linear_scores["test_neg_median_absolute_error"]}")
print()
print(f"Test Neg Mean Squared Error : {linear_scores["test_neg_mean_squared_error"]}")
print()
print(f"Test R2 : {linear_scores["test_r2"]}")

Fit Time : [0.00722933 0.00447154 0.00427818 0.0074904  0.00774288]

Score Time : [0.00579023 0.0064733  0.00621605 0.01547647 0.0106144 ]

Test Neg Median Abosulte Error : [-0.1212494  -0.13782219 -0.13068358 -0.13513305 -0.12558258]

Test Neg Mean Squared Error : [-0.06163943 -0.06789332 -0.03414144 -0.02423649 -0.02716378]

Test R2 : [0.86487471 0.86222643 0.8927639  0.91815104 0.92040313]


### Modèle Lasso

In [24]:
lasso_model = Lasso(alpha=0.2)

lasso_scores = cross_validate(lasso_model, X_train, y_train, scoring=scoring)

print(f"Fit Time : {lasso_scores["fit_time"]}")
print()
print(f"Score Time : {lasso_scores["score_time"]}")
print()
print(f"Test Neg MSE : {lasso_scores["test_neg_mean_squared_error"]}")
print()
print(f"Test Neg  MAE : {lasso_scores["test_neg_median_absolute_error"]}")
print()
print(f"Test Neg SMSE : {lasso_scores["test_neg_root_mean_squared_error"]}")
print()
print(f"Test R2 : {lasso_scores["test_r2"]}")

Fit Time : [0.00690198 0.00782681 0.00600338 0.00777769 0.00868106]

Score Time : [0.00949025 0.00672603 0.00637269 0.01633453 0.0145402 ]

Test Neg MSE : [-0.06189423 -0.0682196  -0.03440771 -0.02494376 -0.02711687]

Test Neg  MAE : [-0.13281945 -0.13966569 -0.12903467 -0.13048909 -0.13127217]

Test Neg SMSE : [-0.24878551 -0.26118882 -0.18549315 -0.15793594 -0.164672  ]

Test R2 : [0.86431615 0.86156432 0.89192756 0.91576252 0.92054061]


### Ridget Model

In [25]:
ridge_model = Ridge(alpha=0.1)
ridge_scores = cross_validate(ridge_model, X_train, y_train, scoring=scoring)

print(f"Fit Time : {ridge_scores["fit_time"]}")
print()
print(f"Score Time : {ridge_scores["score_time"]}")
print()
print(f"Test Neg MSE : {ridge_scores["test_neg_mean_squared_error"]}")
print()
print(f"Test Neg  MAE : {ridge_scores["test_neg_median_absolute_error"]}")
print()
print(f"Test Neg SMSE : {ridge_scores["test_neg_root_mean_squared_error"]}")
print()
print(f"Test R2 : {ridge_scores["test_r2"]}")

Fit Time : [0.03235483 0.00687718 0.02422285 0.00647688 0.00657964]

Score Time : [0.00570917 0.01524568 0.01970172 0.01133108 0.00700164]

Test Neg MSE : [-0.06332884 -0.06075554 -0.03080009 -0.02839163 -0.02887987]

Test Neg  MAE : [-0.11706773 -0.12699329 -0.12899089 -0.11414433 -0.11894166]

Test Neg SMSE : [-0.25165222 -0.24648639 -0.17549955 -0.16849816 -0.16994079]

Test R2 : [0.86117121 0.87671088 0.90325887 0.90411873 0.91537454]


### Arbres de décision

In [26]:
tree_model = DecisionTreeRegressor()
tree_scores = cross_validate(tree_model, X_train, y_train, scoring=scoring)

print(f"Fit Time : {tree_scores["fit_time"]}")
print()
print(f"Score Time : {tree_scores["score_time"]}")
print()
print(f"Test Neg MSE : {tree_scores["test_neg_mean_squared_error"]}")
print()
print(f"Test Neg  MAE : {tree_scores["test_neg_median_absolute_error"]}")
print()
print(f"Test Neg SMSE : {tree_scores["test_neg_root_mean_squared_error"]}")
print()
print(f"Test R2 : {tree_scores["test_r2"]}")

Fit Time : [0.00836873 0.00540471 0.00600743 0.00500703 0.00923967]

Score Time : [0.00534606 0.00564718 0.0062952  0.01005173 0.01365519]

Test Neg MSE : [-0.00102182 -0.00063903 -0.00010957 -0.0001888  -0.00015051]

Test Neg  MAE : [-0.00406088 -0.00712158 -0.0060641  -0.00495831 -0.00393476]

Test Neg SMSE : [-0.03196589 -0.02527898 -0.01046735 -0.01374041 -0.01226833]

Test R2 : [0.99775998 0.99870324 0.99965586 0.99936241 0.99955896]


In [27]:
random_model = RandomForestRegressor()

random_scores = cross_validate(random_model, X_train, y_train, scoring=scoring)

print(f"Fit Time : {random_scores["fit_time"]}")
print()
print(f"Score Time : {random_scores["score_time"]}")
print()
print(f"Test Neg MSE : {random_scores["test_neg_mean_squared_error"]}")
print()
print(f"Test Neg  MAE : {random_scores["test_neg_median_absolute_error"]}")
print()
print(f"Test Neg SMSE : {random_scores["test_neg_root_mean_squared_error"]}")
print()
print(f"Test R2 : {random_scores["test_r2"]}")

Fit Time : [0.2647965  0.21478629 0.22586393 0.20976686 0.21536493]

Score Time : [0.02492237 0.01920319 0.01801109 0.01707602 0.01741076]

Test Neg MSE : [-1.28354516e-03 -1.49280455e-03 -4.23052640e-04 -6.07244602e-05
 -9.00574610e-05]

Test Neg  MAE : [-0.00263888 -0.00332021 -0.0037187  -0.00315089 -0.00358124]

Test Neg SMSE : [-0.0358266  -0.03863683 -0.02056824 -0.00779259 -0.00948986]

Test R2 : [0.99718623 0.9969707  0.99867122 0.99979493 0.99973611]
